# DPLQR epoch experiment: 10 repetitions per case
Extension of the 08 September 2026 (08092026) paired epoch diagnostic.

This experiment uses **10 repetitions for each of Cases 1–3** (30 datasets), with n=1000,
tau=0.5, an 800/200 training/validation split, and the same seven epoch budgets up to 1000.
The six previously verified paths are reused after source, settings, input and version checks;
24 new paths use repetitions 3–10 under the same seed scheme. Each dataset has one uninterrupted
training path, shared by every budget. Seeds change between repetitions; uncertainty here combines
sampling and initialization variability. Multiple initializations on each fixed dataset are a separate study.

The original two-repetition folder and main first-pass/full-replication settings remain untouched.
**Run All** with the repository `.venv` kernel. This notebook is the Python source; no deleted simulation
`.py`, R execution, LQR, PLAQR, or auxiliary-network fits are needed for these coefficient diagnostics.

The three rules are **terminal epoch**, **validation best within budget**, and **patience-15 early
stopping**. The first two continue past early stopping for diagnosis. Truth/test data never select
epochs. Ten repetitions improve the preliminary check but do not provide a full paper replication.

## 1. Imports and reused scientific definitions

In [ ]:
from __future__ import annotations
import ast
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from scipy.stats import norm, t
from sklearn.preprocessing import StandardScaler
import torch
from torchtuples import Model
import torchtuples as tt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p/'dqAux.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open inside the dplqr repository with its .venv kernel.')
PARENT_DIR = ROOT/'results'/'2026-09-04-homoscedastic-simulation'
FIRST_PASS = PARENT_DIR/'first-pass-run'
CACHE_DIR = FIRST_PASS/'2026-09-08-epoch-robustness'
EXPERIMENT_DIR = CACHE_DIR/'repetitions-10'
SOURCE_NOTEBOOK = PARENT_DIR/'simulate_homoscedastic.ipynb'
THIS_NOTEBOOK = EXPERIMENT_DIR/'epoch_robustness.ipynb'
sys.path.insert(0, str(ROOT))
from dqAux import dqNetSparse, checkLoss, checkErrorMean
THETA = np.array([1.0, -1.0])

# Import definitions only; no parent parameter, simulation, or R cells execute.
REUSED_NAMES = {'seed_all', 'nonlinear_truth', 'generate_covariates', 'generate_dataset',
                'as_numpy', 'clip_neural_weights', 'tensor_pair'}
selected_nodes = {}
for cell in nbformat.read(SOURCE_NOTEBOOK, as_version=4).cells:
    if cell.cell_type == 'code':
        for node in ast.parse(cell.source).body:
            if isinstance(node, ast.FunctionDef) and node.name in REUSED_NAMES:
                if node.name in selected_nodes:
                    raise ValueError('Duplicate scientific definition: '+node.name)
                selected_nodes[node.name] = node
assert set(selected_nodes) == REUSED_NAMES, 'Missing parent scientific definitions'
for node in selected_nodes.values():
    module = ast.fix_missing_locations(ast.Module(body=[node], type_ignores=[]))
    exec(compile(module, str(SOURCE_NOTEBOOK), 'exec'), globals())
reference_config = json.loads((FIRST_PASS/'run_config.json').read_text(encoding='utf-8'))
reference_raw = pd.read_csv(FIRST_PASS/'raw_results.csv', float_precision='round_trip')
reference_dplqr = reference_raw.loc[reference_raw.method.eq('DPLQR')].copy()
assert reference_config['status'] == 'complete', 'Complete the first pass first'
print('Python:', sys.executable)
print('Reused:', ', '.join(sorted(REUSED_NAMES)))

## 2. Independent settings
Only repetitions increase. The two-repetition seed sequence is extended to 10; all other
training and data settings stay fixed. Existing outputs are reused only when their code/settings
identity matches. Changed settings require a fresh OUTPUT_DIR. Save edits and restart the kernel
before Run All so the source identity matches executed code.

In [ ]:
EPOCH_BUDGETS = [10, 25, 50, 100, 200, 500, 1000]
PROFILE = dict(repetitions=10, cases=[1, 2, 3], n=1000, test_size=1000, tau=0.5,
               seed=20260904, depth=2, width=32, batch_size=128,
               learning_rate=0.005, patience=15, threads=1)
OUTPUT_DIR = EXPERIMENT_DIR
assert EPOCH_BUDGETS == sorted(set(EPOCH_BUDGETS)) and min(EPOCH_BUDGETS) >= 1
assert 100 in EPOCH_BUDGETS and max(EPOCH_BUDGETS) >= 100
for name in ('test_size', 'seed', 'depth', 'width', 'batch_size',
             'learning_rate', 'patience', 'threads', 'cases'):
    assert PROFILE[name] == reference_config[name], f'{name} must match the saved paired baseline'
assert reference_config['sample_sizes'] == [PROFILE['n']]
assert reference_config['taus'] == [PROFILE['tau']]
assert reference_config['epochs'] == 100 and reference_config['hyperparameter_mode'] == 'fixed'
torch.set_num_threads(PROFILE['threads'])
torch.use_deterministic_algorithms(True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Epoch budgets:', EPOCH_BUDGETS)
print('Fixed paired profile:', PROFILE)
print('Outputs:', OUTPUT_DIR)
assert PROFILE['repetitions'] >= reference_config['repetitions']
EXPECTED_TRAJECTORIES = len(PROFILE['cases'])*PROFILE['repetitions']
print('Total paired trajectories:', EXPECTED_TRAJECTORIES)

## 3. Data generation and paired inputs
Repetitions 1–2 are verified against the saved first-pass exports. Repetitions 3–10 are new independent
draws from the same generator/seed scheme. Every repetition's training, validation and test arrays
are saved in `paired_datasets/` with numerical hashes, so all epoch comparisons use identical data.
Training observations alone determine standardization. Regenerated arrays must match on resume.

In [ ]:
def paired_data(case, rep):
    n = PROFILE['n']
    setting_seed = PROFILE['seed'] + case*10_000_000 + n*1_000 + rep
    rng = np.random.default_rng(setting_seed)
    x, z, y, _ = generate_dataset(n, case, rng)
    order = rng.permutation(n)
    tr, va = order[:int(0.8*n)], order[int(0.8*n):]
    xt, zt, yt, mt = generate_dataset(PROFILE['test_size'], case, rng)
    stem = f'case_{case}_n_{n}_rep_{rep:04d}'
    checks = []
    for split, expected in [
        ('train', np.column_stack((y[tr], x[tr], z[tr], nonlinear_truth(z[tr], case)))),
        ('test', np.column_stack((yt, xt, zt, mt)))]:
        path = FIRST_PASS/'data_for_r'/f'{stem}_{split}.csv.gz'
        difference = None
        if rep <= reference_config['repetitions']:
            saved = pd.read_csv(path, float_precision='round_trip').to_numpy()
            np.testing.assert_allclose(saved, expected, rtol=0, atol=1e-12)
            difference = float(np.max(np.abs(saved-expected)))
        checks.append(dict(case=case, rep=rep, split=split,
                           verified_against_first_pass=rep <= reference_config['repetitions'],
                           max_abs_difference=difference,
                           array_sha256=hashlib.sha256(np.ascontiguousarray(expected).tobytes()).hexdigest()))
    data = dict(xtr=x[tr], ztr=z[tr], ytr=y[tr], xv=x[va], zv=z[va], yv=y[va],
                xt=xt, zt=zt, yt=yt, true_m=mt+t.ppf(PROFILE['tau'], df=3),
                network_seed=setting_seed+int(round(PROFILE['tau']*1_000_000)))
    data_dir = OUTPUT_DIR/'paired_datasets'
    data_dir.mkdir(exist_ok=True)
    data_path = data_dir/f'{stem}.npz'
    if data_path.exists():
        with np.load(data_path, allow_pickle=False) as saved:
            assert set(saved.files) == set(data)
            for name,value in data.items():
                np.testing.assert_array_equal(saved[name],value)
    else:
        temporary = data_path.with_suffix('.tmp')
        with temporary.open('wb') as stream:
            np.savez_compressed(stream, **data)
        temporary.replace(data_path)
    checks.append(dict(case=case, rep=rep, split='all_train_validation_test_arrays',
                       verified_against_first_pass=False, max_abs_difference=None,
                       array_sha256=hashlib.sha256(b''.join(np.asarray(data[k]).tobytes()
                                                          for k in sorted(data))).hexdigest()))
    return data, checks

## 4. Observe one uninterrupted training path
Construction, initialization, loss, optimizer and `fit` arguments follow the parent's
`train_dplqr_once`. The observer never stops or modifies the training network. It clips an **evaluation
clone** for endpoint diagnostics, matching the parent's final clipping. Direct forward passes avoid
prediction DataLoaders, which can consume RNG and change subsequent training.

`selection_val_loss` preserves the original torchtuples **pre-clipping minibatch-averaged** validation
score for selecting epochs. Diagnostic train/validation/test losses use full-sample means after
clipping. They need not equal the selection score. Epoch 0 records initialization and is not selectable.

In [ ]:
class EpochRecorder(tt.callbacks.Callback):
    def __init__(self, evaluator, inputs, outcomes, true_m, test_x, case, rep):
        self.evaluator, self.inputs, self.outcomes = evaluator, inputs, outcomes
        self.true_m, self.test_x = true_m, test_x
        self.case, self.rep = case, rep
        self.rows = []

    def observe(self, epoch, score):
        rng_before = torch.random.get_rng_state().clone()
        self.evaluator.load_state_dict(self.model.net.state_dict())
        clip_neural_weights(self.evaluator)
        self.evaluator.eval()
        with torch.no_grad():
            theta = self.evaluator.linLinear.weight.detach().cpu().numpy().reshape(-1).copy()
            predictions = {name: self.evaluator(*pair).cpu().numpy().reshape(-1)
                           for name, pair in self.inputs.items()}
        assert torch.equal(rng_before, torch.random.get_rng_state()), 'Observer changed Torch RNG'
        losses = {name+'_check_loss': float(checkErrorMean(pred[:, None], self.outcomes[name][:, None],
                                                           tau=PROFILE['tau']))
                  for name, pred in predictions.items()}
        m_hat = predictions['test'] - self.test_x @ theta
        row = dict(case=self.case, n=PROFILE['n'], rep=self.rep, tau=PROFILE['tau'], epoch=epoch,
                   theta1=float(theta[0]), theta2=float(theta[1]),
                   error_theta1=float(theta[0]-THETA[0]), error_theta2=float(theta[1]-THETA[1]),
                   selection_val_loss=score,
                   relative_mse=float(np.mean((m_hat-self.true_m)**2)/np.mean(self.true_m**2)), **losses)
        assert np.isfinite([v for k,v in row.items() if k != 'selection_val_loss']).all()
        assert epoch == 0 or np.isfinite(score)
        self.rows.append(row)

    def on_fit_start(self):
        self.observe(0, np.nan)

    def on_epoch_end(self):
        epoch = len(self.rows)
        self.observe(epoch, float(self.model.val_metrics.scores['loss']['score'][-1]))
        if epoch % 250 == 0:
            print(f'case={self.case}, rep={self.rep}, epoch={epoch}', flush=True)
        return False

def train_epoch_path(data, case, rep):
    seed_all(data['network_seed'])
    scaler = StandardScaler().fit(data['ztr'])
    inputs = {'train': tensor_pair(data['xtr'], scaler.transform(data['ztr'])),
              'validation': tensor_pair(data['xv'], scaler.transform(data['zv'])),
              'test': tensor_pair(data['xt'], scaler.transform(data['zt']))}
    net = dqNetSparse(2, 8, torch.zeros((1, 2), dtype=torch.float32),
                      [PROFILE['depth'], PROFILE['width']], sparseRatio=0.5)
    net.linLinear.reset_parameters()
    model = Model(net, checkLoss(tau=PROFILE['tau']), device='cpu')
    model.optimizer.set_lr(PROFILE['learning_rate'])
    observer = EpochRecorder(copy.deepcopy(net).requires_grad_(False), inputs,
                             {'train': data['ytr'], 'validation': data['yv'], 'test': data['yt']},
                             data['true_m'], data['xt'], case, rep)
    model.fit(inputs['train'], torch.tensor(data['ytr'][:, None], dtype=torch.float32),
              PROFILE['batch_size'], max(EPOCH_BUDGETS), [observer], False,
              val_data=(inputs['validation'], torch.tensor(data['yv'][:, None], dtype=torch.float32)),
              val_batch_size=PROFILE['batch_size'])
    frame = pd.DataFrame(observer.rows)
    assert frame.epoch.tolist() == list(range(max(EPOCH_BUDGETS)+1))
    return frame

def choose_epoch(trace, budget, rule):
    eligible = trace.loc[trace.epoch.between(1, budget)].sort_values('epoch')
    if rule == 'terminal':
        return eligible.iloc[-1], budget
    best, since_best, stopped_at = None, 0, budget
    for row in eligible.itertuples(index=False):
        if best is None or row.selection_val_loss < best.selection_val_loss:
            best, since_best = row, 0
        else:
            since_best += 1
        if rule == 'patience15' and since_best >= PROFILE['patience']:
            stopped_at = row.epoch
            break
    return trace.loc[trace.epoch.eq(best.epoch)].iloc[0], int(stopped_at)

def atomic_csv(frame, path):
    temporary = path.with_suffix('.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def experiment_identity():
    nb = nbformat.read(THIS_NOTEBOOK, as_version=4)
    source = '\n'.join(ast.dump(ast.parse(c.source), include_attributes=False)
                        for c in nb.cells if c.cell_type == 'code')
    reused = {name: ast.dump(node, include_attributes=False) for name, node in selected_nodes.items()}
    return dict(profile=PROFILE, budgets=EPOCH_BUDGETS,
                code_sha256=hashlib.sha256(source.encode()).hexdigest(),
                scientific_functions_sha256=hashlib.sha256(json.dumps(reused, sort_keys=True).encode()).hexdigest(),
                dqAux_sha256=hashlib.sha256((ROOT/'dqAux.py').read_bytes()).hexdigest(),
                baseline_config_sha256=hashlib.sha256((FIRST_PASS/'run_config.json').read_bytes()).hexdigest(),
                baseline_results_sha256=hashlib.sha256((FIRST_PASS/'raw_results.csv').read_bytes()).hexdigest(),
                versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                              torch=torch.__version__, torchtuples=tt.__version__))


def verify_cache_compatibility(identity):
    previous = json.loads((CACHE_DIR/'epoch_run_config.json').read_text(encoding='utf-8'))
    assert previous['status'] == 'complete' and previous['baseline_verified']
    old = previous['identity']
    for name in ('scientific_functions_sha256','dqAux_sha256','baseline_config_sha256',
                 'baseline_results_sha256','versions','budgets'):
        assert old[name] == identity[name], f'Cached paths incompatible: {name}'
    assert old['profile']['repetitions'] == reference_config['repetitions']
    assert {k:v for k,v in old['profile'].items() if k!='repetitions'} == {
        k:v for k,v in identity['profile'].items() if k!='repetitions'}
    old_nb = nbformat.read(CACHE_DIR/'epoch_robustness.ipynb',as_version=4)
    new_nb = nbformat.read(THIS_NOTEBOOK,as_version=4)
    source = '\n'.join(ast.dump(ast.parse(c.source),include_attributes=False)
                       for c in old_nb.cells if c.cell_type=='code')
    assert hashlib.sha256(source.encode()).hexdigest() == old['code_sha256']
    def training_definitions(notebook):
        return {node.name:ast.dump(node,include_attributes=False)
                for cell in notebook.cells if cell.cell_type=='code'
                for node in ast.parse(cell.source).body
                if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and
                node.name in ('EpochRecorder','train_epoch_path','choose_epoch')}
    assert training_definitions(old_nb) == training_definitions(new_nb), 'Training implementation changed'
    paths = {f'case_{case}_rep_{rep:04d}.csv':
             hashlib.sha256((CACHE_DIR/'trajectories'/f'case_{case}_rep_{rep:04d}.csv').read_bytes()).hexdigest()
             for case,rep in itertools.product(PROFILE['cases'],range(1,old['profile']['repetitions']+1))}
    return dict(source_config_sha256=hashlib.sha256((CACHE_DIR/'epoch_run_config.json').read_bytes()).hexdigest(),
                source_code_sha256=old['code_sha256'], trajectory_hashes=paths)

def validate_trace(trace, case, rep):
    assert trace.epoch.tolist() == list(range(max(EPOCH_BUDGETS)+1))
    assert trace.case.eq(case).all() and trace.rep.eq(rep).all()
    assert trace.n.eq(PROFILE['n']).all() and trace.tau.eq(PROFILE['tau']).all()
    assert np.isfinite(trace.drop(columns='selection_val_loss').to_numpy()).all()
    assert np.isfinite(trace.loc[trace.epoch.gt(0),'selection_val_loss']).all()
    for j in (1,2):
        np.testing.assert_allclose(trace[f'error_theta{j}'],trace[f'theta{j}']-THETA[j-1],atol=1e-14,rtol=0)


## 5. Run 30 trajectories (six reused; 24 new)
Cache reuse requires matching scientific functions, training definitions, package versions, data
settings, seeds and epoch budgets. Repetitions 1–2 additionally reproduce the six original saved
cap-100/patience-15 fits. New repetitions have no original first-pass fit to compare; their early-stop
baselines are reconstructed from the same continuous path and are reported without claiming an
external baseline match. Every trajectory is checkpointed and has a provenance row.

In [ ]:
def run_experiment():
    identity = experiment_identity()
    cache_provenance = verify_cache_compatibility(identity)
    identity['cache_provenance'] = cache_provenance
    config_path = OUTPUT_DIR/'epoch_run_config.json'
    trajectories_dir = OUTPUT_DIR/'trajectories'
    trajectories_dir.mkdir(exist_ok=True)
    if config_path.exists():
        previous = json.loads(config_path.read_text(encoding='utf-8'))
        if previous['identity'] != identity:
            raise ValueError('Settings/source changed. Choose a fresh OUTPUT_DIR.')
    elif list(trajectories_dir.glob('*.csv')):
        raise ValueError('Existing trajectories lack configuration. Choose a fresh OUTPUT_DIR.')
    started = time.perf_counter()
    metadata = dict(created_utc=datetime.now(timezone.utc).isoformat(), identity=identity, status='running',
                    completed_trajectories=0, expected_trajectories=EXPECTED_TRAJECTORIES,
                    reused_original_trajectories=0, newly_fitted_trajectories=0, resumed_trajectories=0,
                    early_stopping='Virtual only; live training continues to the maximum epoch',
                    inference='Coefficient/prediction diagnostics; no SEs, coverage, auxiliary networks or R')
    traces, data_checks, baseline_checks, budget_rows, provenance = [], [], [], [], []
    config_path.write_text(json.dumps(metadata, indent=2),encoding='utf-8')
    try:
        for case,rep in itertools.product(PROFILE['cases'],range(1,PROFILE['repetitions']+1)):
            data,checks = paired_data(case,rep)
            data_checks.extend(checks)
            path = trajectories_dir/f'case_{case}_rep_{rep:04d}.csv'
            if path.exists():
                trace = pd.read_csv(path,float_precision='round_trip')
                origin = 'resumed_this_experiment'
                metadata['resumed_trajectories'] += 1
            elif path.name in cache_provenance['trajectory_hashes']:
                source = CACHE_DIR/'trajectories'/path.name
                assert hashlib.sha256(source.read_bytes()).hexdigest() == cache_provenance['trajectory_hashes'][path.name]
                shutil.copy2(source,path)
                trace = pd.read_csv(path,float_precision='round_trip')
                origin = 'verified_two_repetition_cache'
                metadata['reused_original_trajectories'] += 1
            else:
                trace = train_epoch_path(data,case,rep)
                atomic_csv(trace,path)
                origin = 'newly_trained'
                metadata['newly_fitted_trajectories'] += 1
            validate_trace(trace,case,rep)
            baseline,stop = choose_epoch(trace,100,'patience15')
            saved = reference_dplqr.loc[reference_dplqr.case.eq(case)&reference_dplqr.rep.eq(rep)]
            if not saved.empty:
                assert len(saved)==1
                saved = saved.iloc[0]
                np.testing.assert_allclose(baseline[['theta1','theta2']].to_numpy(dtype=float),
                                           saved[['theta1','theta2']].to_numpy(dtype=float),rtol=0,atol=1e-7)
                np.testing.assert_allclose(baseline.relative_mse,saved.rmse_m,rtol=0,atol=1e-6)
                np.testing.assert_allclose(baseline.test_check_loss,saved.test_check_loss,rtol=0,atol=1e-6)
                assert stop == saved.epochs_run
                baseline_checks.append(dict(case=case,rep=rep,selected_epoch=int(baseline.epoch),
                    actual_stop_epoch=stop,theta1_difference=float(baseline.theta1-saved.theta1),
                    theta2_difference=float(baseline.theta2-saved.theta2),
                    relative_mse_difference=float(baseline.relative_mse-saved.rmse_m),passed=True))
            for budget,rule in itertools.product(EPOCH_BUDGETS,['terminal','validation_best','patience15']):
                selected,stopped_at = choose_epoch(trace,budget,rule)
                row = selected.to_dict()
                row.pop('epoch')
                row.update(epoch_budget=budget,rule=rule,selected_epoch=int(selected.epoch),epochs_trained=stopped_at)
                budget_rows.append(row)
            traces.append(trace)
            provenance.append(dict(case=case,rep=rep,origin=origin,network_seed=data['network_seed'],
                                   sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
            atomic_csv(pd.DataFrame(provenance),OUTPUT_DIR/'trajectory_provenance.csv')
            metadata.update(completed_trajectories=len(traces),elapsed_seconds=time.perf_counter()-started,
                            last_completed_case=case,last_completed_rep=rep)
            config_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
            print(f'Completed {len(traces)}/{EXPECTED_TRAJECTORIES}: case={case}, rep={rep}, {origin}; '
                  f'elapsed={metadata["elapsed_seconds"]:.1f}s',flush=True)
        raw,budgets = pd.concat(traces,ignore_index=True),pd.DataFrame(budget_rows)
        for frame in (raw,budgets):
            for name in ('case','n','rep'):
                frame[name] = frame[name].astype(int)
        assert len(raw) == EXPECTED_TRAJECTORIES*(max(EPOCH_BUDGETS)+1)
        assert len(budgets) == EXPECTED_TRAJECTORIES*len(EPOCH_BUDGETS)*3
        assert len(baseline_checks) == len(reference_dplqr)
        for frame,name in [(raw,'epoch_trajectories'),(budgets,'epoch_budget_results'),
                            (pd.DataFrame(data_checks),'data_pairing_checks'),
                            (pd.DataFrame(baseline_checks),'baseline_reproduction_checks')]:
            atomic_csv(frame,OUTPUT_DIR/f'{name}.csv')
        metadata.update(status='complete',trajectory_rows=len(raw),budget_rows=len(budgets),
                        existing_baselines_verified=len(baseline_checks),elapsed_seconds=time.perf_counter()-started)
        config_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
        return raw,budgets
    except Exception as exc:
        metadata.update(status='failed',error=repr(exc),elapsed_seconds=time.perf_counter()-started)
        config_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
        raise

trajectories,budget_results = run_experiment()
display(pd.read_csv(OUTPUT_DIR/'trajectory_provenance.csv'))
display(pd.read_csv(OUTPUT_DIR/'baseline_reproduction_checks.csv'))

## 6. Summaries and paired drift
Bias is the average signed coefficient error over **10 repetitions**; SD uses ddof=1. Bias MCSE is
SD/sqrt(10), expressing the simulation uncertainty of the estimated bias (not an individual fit's SE).
Mean absolute error avoids cancellation. Drift comparisons are paired within each dataset between
terminal epochs 100 and 1000; a positive absolute-error change means movement farther from truth.

In [ ]:
summary_rows = []
for (case,budget,rule), group in budget_results.groupby(['case','epoch_budget','rule']):
    row = dict(case=case, n=PROFILE['n'], tau=PROFILE['tau'], epoch_budget=budget, rule=rule,
               repetitions=len(group), mean_selected_epoch=group.selected_epoch.mean(),
               mean_relative_mse=group.relative_mse.mean(), mean_train_check_loss=group.train_check_loss.mean(),
               mean_validation_check_loss=group.validation_check_loss.mean(),
               mean_test_check_loss=group.test_check_loss.mean())
    for j in (1,2):
        error = group[f'theta{j}']-THETA[j-1]
        row.update({f'mean_theta{j}':group[f'theta{j}'].mean(), f'bias_theta{j}':error.mean(),
                    f'sd_theta{j}':group[f'theta{j}'].std(ddof=1),
                    f'mae_theta{j}':error.abs().mean(), f'mse_theta{j}':(error**2).mean()})
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
assert summary.repetitions.eq(PROFILE['repetitions']).all()
for j in (1,2):
    summary[f'mcse_bias_theta{j}'] = summary[f'sd_theta{j}']/np.sqrt(summary.repetitions)
summary.to_csv(OUTPUT_DIR/'epoch_summary.csv', index=False)
coefficient_table = summary[['case','n','tau','rule','epoch_budget']].copy()
for j in (1,2):
    coefficient_table[f'theta{j}_bias_sd'] = [f'{b:.4f} ({s:.4f})' for b,s in
                                             zip(summary[f'bias_theta{j}'],summary[f'sd_theta{j}'])]
coefficient_table['relative_mse'] = summary.mean_relative_mse
coefficient_table.to_csv(OUTPUT_DIR/'coefficient_bias_sd_by_epoch.csv',index=False)
terminal = budget_results.loc[budget_results.rule.eq('terminal')]
start = terminal.loc[terminal.epoch_budget.eq(100)].set_index(['case','rep'])
end = terminal.loc[terminal.epoch_budget.eq(max(EPOCH_BUDGETS))].set_index(['case','rep'])
drift = start[['n','tau']].copy()
for j in (1,2):
    drift[f'theta{j}_at_100'] = start[f'theta{j}']
    drift[f'theta{j}_at_final'] = end[f'theta{j}']
    drift[f'theta{j}_change'] = end[f'theta{j}']-start[f'theta{j}']
    drift[f'absolute_error_change_theta{j}'] = (end[f'theta{j}']-THETA[j-1]).abs()-(start[f'theta{j}']-THETA[j-1]).abs()
drift['relative_mse_change'] = end.relative_mse-start.relative_mse
drift['test_check_loss_change'] = end.test_check_loss-start.test_check_loss
drift = drift.reset_index()
drift.to_csv(OUTPUT_DIR/'paired_drift_100_to_final.csv',index=False)
display(coefficient_table.loc[coefficient_table.epoch_budget.isin([100,500,1000])])
display(drift)

paired_summary_rows = []
for case,g in drift.groupby('case'):
    row = dict(case=case,repetitions=len(g))
    for j in (1,2):
        delta = g[f'absolute_error_change_theta{j}']
        row.update({f'mean_absolute_error_change_theta{j}':delta.mean(),
                    f'mcse_absolute_error_change_theta{j}':delta.std(ddof=1)/np.sqrt(len(delta)),
                    f'farther_from_truth_theta{j}':int(delta.gt(0).sum())})
    row['test_loss_worse'] = int(g.test_check_loss_change.gt(0).sum())
    row['nuisance_mse_worse'] = int(g.relative_mse_change.gt(0).sum())
    paired_summary_rows.append(row)
paired_drift_summary = pd.DataFrame(paired_summary_rows)
paired_drift_summary.to_csv(OUTPUT_DIR/'paired_drift_summary.csv',index=False)
display(paired_drift_summary)


## 7. Figures
All ten paths per case are shown, together with their mean and the true coefficient.
Bias and absolute-error panels compare checkpoint-selection rules. PNG and vector PDF outputs
are saved in `figures/`. Monte Carlo SEs of bias are in `epoch_summary.csv`.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "semibold", "axes.labelcolor": "#263238",
    "text.color": "#263238", "axes.edgecolor": "#9aa4aa",
    "grid.color": "#dbe1e5", "grid.linewidth": 0.6,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
case_order = list(PROFILE["cases"])
rep_order = sorted(trajectories["rep"].unique())
rule_styles = {
    "terminal": ("#2878b5", "-", "o", "At the final epoch"),
    "validation_best": ("#7b55a3", "--", "s", "Best validation epoch within budget"),
    "patience15": ("#168375", "-.", "^", "Early stopping: patience 15"),
}
diagnostic_curves = trajectories.loc[trajectories["epoch"] >= 1].copy()
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
figure_paths = []
repetition_label = f"{PROFILE['repetitions']} repetitions per case"


def epoch_axis(ax, budgets_only=False):
    ax.axvline(100, color="#939a9e", linestyle=":", linewidth=0.8)
    ax.set_xscale("log")
    ticks = list(EPOCH_BUDGETS)
    if not budgets_only and 1 not in ticks:
        ticks = [1] + ticks
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xlim(min(ticks), max(ticks))
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(True, which="major", alpha=0.75)
    ax.set_axisbelow(True)


def save_diagnostic(fig, stem):
    fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
    figure_paths.extend([FIGURE_DIR / f"{stem}.png", FIGURE_DIR / f"{stem}.pdf"])
    display(fig)
    plt.close(fig)


fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 9.6), squeeze=False)
for row_index, case in enumerate(case_order):
    case_data = diagnostic_curves.loc[diagnostic_curves["case"] == case]
    averaged = case_data.groupby("epoch", as_index=False)[["theta1", "theta2"]].mean()
    baseline = budget_results.loc[
        budget_results["case"].eq(case)
        & budget_results["rule"].eq("patience15")
        & budget_results["epoch_budget"].eq(100)
    ]
    assert len(baseline) == PROFILE["repetitions"]
    for column, component in enumerate((1, 2)):
        ax = axes[row_index, column]
        for rep in rep_order:
            path = case_data.loc[case_data["rep"] == rep].sort_values("epoch")
            ax.plot(path["epoch"], path[f"theta{component}"],
                    color="#8495a2", linewidth=0.75, alpha=0.43)
        ax.plot(averaged["epoch"], averaged[f"theta{component}"],
                color="#2878b5", linewidth=2.0)
        ax.axhline(float(baseline[f"theta{component}"].mean()),
                   color="#168375", linestyle=":", linewidth=1.8)
        ax.axhline(float(THETA[column]), color="#222222", linestyle="--", linewidth=1.2)
        ax.set_title(f"Case {case} | $\\theta_{component}$", loc="left", fontsize=11)
        ax.set_ylabel("Coefficient estimate")
        epoch_axis(ax)
        if row_index == len(case_order) - 1:
            ax.set_xlabel("Completed training epochs (log scale)")
handles = [
    Line2D([0], [0], color="#8495a2", alpha=0.6, label="Individual training path"),
    Line2D([0], [0], color="#2878b5", linewidth=2, label="Mean of training paths"),
    Line2D([0], [0], color="#222222", linestyle="--", label="True coefficient"),
    Line2D([0], [0], color="#168375", linestyle=":", label="Mean: patience 15 / cap 100"),
]
fig.suptitle("Do the DPLQR coefficients drift as training continues?", fontsize=15, y=0.98)
fig.text(0.5, 0.945, "Early stopping is disabled for the paths; the dotted reference retains the first-pass stopping rule.",
         ha="center", fontsize=10)
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.035), ncol=2, frameon=False)
fig.text(0.5, 0.010, f"{repetition_label}; n = 1,000, median quantile. Each path keeps its data and initialization fixed.",
         ha="center", fontsize=9, color="#52636b")
fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.155, hspace=0.38, wspace=0.25)
save_diagnostic(fig, "theta_trajectories")


for statistic, ylabel, title, subtitle, stem in [
    ("bias", "Mean signed error", "Mean coefficient error by epoch budget",
     "Signed error is the average estimate minus truth; opposite errors can cancel.",
     "bias_by_epoch_and_rule"),
    ("mae", "Mean absolute error", "Coefficient accuracy by epoch budget",
     "Absolute errors avoid cancellation across repetitions; lower values indicate closer estimates.",
     "absolute_error_by_epoch_and_rule"),
]:
    fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 9.6), squeeze=False)
    for row_index, case in enumerate(case_order):
        for column, component in enumerate((1, 2)):
            ax = axes[row_index, column]
            for rule, (color, linestyle, marker, label) in rule_styles.items():
                values = summary.loc[(summary["case"] == case) & (summary["rule"] == rule)].sort_values("epoch_budget")
                ax.plot(values["epoch_budget"], values[f"{statistic}_theta{component}"],
                        color=color, linestyle=linestyle, marker=marker, markersize=4,
                        linewidth=1.6, label=label)
            ax.axhline(0, color="#5d6770", linestyle=":", linewidth=1)
            if statistic == "mae":
                ax.set_ylim(bottom=0)
            ax.set_title(f"Case {case} | $\\theta_{component}$", loc="left", fontsize=11)
            ax.set_ylabel(ylabel)
            epoch_axis(ax, budgets_only=True)
            if row_index == len(case_order) - 1:
                ax.set_xlabel("Maximum epoch budget (log scale)")
    fig.suptitle(title, fontsize=15, y=0.98)
    fig.text(0.5, 0.945, subtitle, ha="center", fontsize=10)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.035), ncol=3, frameon=False)
    fig.text(0.5, 0.012, f"Exploratory means from {repetition_label}; Monte Carlo standard errors of bias are in the CSV summary.",
             ha="center", fontsize=9, color="#52636b")
    fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.135, hspace=0.38, wspace=0.25)
    save_diagnostic(fig, stem)


fig, axes = plt.subplots(len(case_order), 2, figsize=(12.5, 10), squeeze=False)
loss_styles = {
    "train_check_loss": ("#2878b5", "-", "Training check loss"),
    "validation_check_loss": ("#d97732", "--", "Validation check loss"),
    "test_check_loss": ("#525c67", "-.", "Test check loss"),
}
for row_index, case in enumerate(case_order):
    averaged = diagnostic_curves.loc[diagnostic_curves["case"] == case].groupby("epoch", as_index=False).agg(
        train_check_loss=("train_check_loss", "mean"),
        validation_check_loss=("validation_check_loss", "mean"),
        test_check_loss=("test_check_loss", "mean"),
    )
    ax = axes[row_index, 0]
    for field, (color, linestyle, label) in loss_styles.items():
        ax.plot(averaged["epoch"], averaged[field], color=color, linestyle=linestyle,
                linewidth=1.3, label=label)
    ax.set_title(f"Case {case} | Prediction loss at each epoch", loc="left", fontsize=11)
    ax.set_ylabel("Mean check loss")
    epoch_axis(ax)
    ax = axes[row_index, 1]
    for rule, (color, linestyle, marker, label) in rule_styles.items():
        values = summary.loc[(summary["case"] == case) & (summary["rule"] == rule)].sort_values("epoch_budget")
        ax.plot(values["epoch_budget"], values["mean_relative_mse"],
                color=color, linestyle=linestyle, marker=marker, markersize=4,
                linewidth=1.6, label=label)
    ax.set_title(f"Case {case} | Nuisance-function error by budget", loc="left", fontsize=11)
    ax.set_ylabel("Mean relative MSE")
    ax.set_ylim(bottom=0)
    epoch_axis(ax, budgets_only=True)
    if row_index == len(case_order) - 1:
        axes[row_index, 0].set_xlabel("Completed training epochs (log scale)")
        axes[row_index, 1].set_xlabel("Maximum epoch budget (log scale)")
fig.suptitle("Does prediction improve while the coefficients move?", fontsize=15, y=0.98)
fig.text(0.5, 0.946, f"Means of {PROFILE['repetitions']} repetitions. Relative MSE uses the paper's squared-error ratio, without a square root.",
         ha="center", fontsize=10)
loss_handles, loss_labels = axes[0, 0].get_legend_handles_labels()
rule_handles, rule_labels = axes[0, 1].get_legend_handles_labels()
fig.legend(loss_handles, loss_labels, loc="lower center", bbox_to_anchor=(0.5, 0.059), ncol=3, frameon=False)
fig.legend(rule_handles, rule_labels, loc="lower center", bbox_to_anchor=(0.5, 0.032), ncol=3, frameon=False)
fig.text(0.5, 0.010, "Test loss and known truth are diagnostics only; neither is used to select the epoch.",
         ha="center", fontsize=9, color="#52636b")
fig.subplots_adjust(left=0.08, right=0.985, top=0.895, bottom=0.157, hspace=0.38, wspace=0.25)
save_diagnostic(fig, "prediction_diagnostics")
print(f"Saved {len(figure_paths)} figure files (four PNGs and four PDFs) to {FIGURE_DIR}")

## 8. Results report

In [ ]:
def markdown_table(frame):
    lines = ['| '+' | '.join(map(str, frame.columns))+' |',
             '| '+' | '.join(['---']*len(frame.columns))+' |']
    for values in frame.itertuples(index=False, name=None):
        lines.append('| '+' | '.join(map(str, values))+' |')
    return '\n'.join(lines)


last = max(EPOCH_BUDGETS)
labels = {'terminal': 'Terminal epoch', 'validation_best': 'Validation best', 'patience15': 'Patience 15'}
rows = []
for case, rule, budget in itertools.product(PROFILE['cases'], labels, sorted(set([100, 500, last]))):
    if budget not in EPOCH_BUDGETS:
        continue
    g = summary.loc[summary.case.eq(case) & summary.rule.eq(rule) & summary.epoch_budget.eq(budget)].iloc[0]
    rows.append({'Case': case, 'Rule': labels[rule], 'Epoch cap': budget,
                 'theta1 bias (SD)': f'{g.bias_theta1:.4f} ({g.sd_theta1:.4f})',
                 'theta1 MAE': f'{g.mae_theta1:.4f}',
                 'theta2 bias (SD)': f'{g.bias_theta2:.4f} ({g.sd_theta2:.4f})',
                 'theta2 MAE': f'{g.mae_theta2:.4f}',
                 'Nuisance relative MSE': f'{g.mean_relative_mse:.4f}'})
farther1 = int(drift.absolute_error_change_theta1.gt(0).sum())
farther2 = int(drift.absolute_error_change_theta2.gt(0).sum())
test_worse = int(drift.test_check_loss_change.gt(0).sum())
metadata = json.loads((OUTPUT_DIR/'epoch_run_config.json').read_text(encoding='utf-8'))
baseline_checks = pd.read_csv(OUTPUT_DIR/'baseline_reproduction_checks.csv')
verified_original = len(baseline_checks)
old_metadata = json.loads((CACHE_DIR/'epoch_run_config.json').read_text(encoding='utf-8'))
cached_count = len(PROFILE['cases']) * old_metadata['identity']['profile']['repetitions']
new_count = EXPECTED_TRAJECTORIES - cached_count
lines = ['# DPLQR epoch experiment: ten repetitions per case', '',
    'Created 08 September 2026 (08092026).', '',
    f'Completed **{EXPECTED_TRAJECTORIES} paired trajectories** through {last} epochs: '
    f'Cases 1–3, n=1000, tau=0.5, **{PROFILE["repetitions"]} repetitions per case**. '
    f'The extension reuses {cached_count} verified trajectories from the earlier two-repetition '
    f'experiment and adds {new_count} new trajectories. The execution recorded '
    f'{metadata["elapsed_seconds"]:.1f} seconds for the run stage, including cache checks. '
    'The main first-pass notebook, full-replication settings, and earlier results remain unchanged.', '',
    '## Findings', '',
    f'From terminal epoch 100 to {last}, absolute theta1 error increased in '
    f'**{farther1}/{EXPECTED_TRAJECTORIES}** datasets, absolute theta2 error increased in '
    f'**{farther2}/{EXPECTED_TRAJECTORIES}**, and test check loss increased in '
    f'**{test_worse}/{EXPECTED_TRAJECTORIES}**. These compare continued training on the same data '
    'and starting weights. Smaller signed bias alone need not mean smaller individual errors.', '']
for case in PROFILE['cases']:
    a = summary.loc[summary.case.eq(case) & summary.rule.eq('terminal') & summary.epoch_budget.eq(100)].iloc[0]
    b = summary.loc[summary.case.eq(case) & summary.rule.eq('terminal') & summary.epoch_budget.eq(last)].iloc[0]
    changes = drift.loc[drift.case.eq(case)]
    farther_case = int(changes.absolute_error_change_theta1.gt(0).sum())
    lines.append(f'- Case {case}: theta1 bias {a.bias_theta1:.4f} → {b.bias_theta1:.4f}; '
                 f'mean absolute theta1 error {a.mae_theta1:.4f} → {b.mae_theta1:.4f}; '
                 f'absolute theta1 error increases in {farther_case}/{len(changes)} datasets. '
                 f'Theta2 bias {a.bias_theta2:.4f} → {b.bias_theta2:.4f}; '
                 f'mean absolute theta2 error {a.mae_theta2:.4f} → {b.mae_theta2:.4f}.')
unchanged = 0
for case, rep in itertools.product(PROFILE['cases'], range(1, PROFILE['repetitions']+1)):
    g = budget_results.loc[budget_results.case.eq(case) & budget_results.rep.eq(rep)
                           & budget_results.rule.eq('patience15')]
    unchanged += int(g.loc[g.epoch_budget.eq(100), 'selected_epoch'].iloc[0]
                     == g.loc[g.epoch_budget.eq(last), 'selected_epoch'].iloc[0])
lines += ['', f'With patience-15 stopping, the selected epoch is unchanged between caps 100 and {last} '
          f'in **{unchanged}/{EXPECTED_TRAJECTORIES}** datasets. Raising the maximum cap therefore '
          'has a different effect from forcing training to continue. Stable selection does not '
          'itself establish accurate coefficients.', '',
          '## How much did the two-repetition estimate change?', '',
          'The table compares terminal-epoch signed bias in the original two repetitions with '
          'the expanded ten repetitions. The ten include the original two; these are nested '
          'summaries, not independent experiments.', '']
old_summary = pd.read_csv(CACHE_DIR/'epoch_summary.csv', float_precision='round_trip')
comparison_rows = []
for case, budget in itertools.product(PROFILE['cases'], [100, last]):
    a = old_summary.loc[old_summary.case.eq(case) & old_summary.rule.eq('terminal')
                        & old_summary.epoch_budget.eq(budget)].iloc[0]
    b = summary.loc[summary.case.eq(case) & summary.rule.eq('terminal')
                    & summary.epoch_budget.eq(budget)].iloc[0]
    comparison_rows.append({'Case': case, 'Terminal epoch': budget,
        'theta1 bias: 2 reps': f'{a.bias_theta1:.4f}', 'theta1 bias: 10 reps': f'{b.bias_theta1:.4f}',
        'theta2 bias: 2 reps': f'{a.bias_theta2:.4f}', 'theta2 bias: 10 reps': f'{b.bias_theta2:.4f}'})
lines += [markdown_table(pd.DataFrame(comparison_rows)), '', '## Figures', '']
for title, name in [('Coefficient trajectories', 'theta_trajectories'),
                    ('Bias by epoch and rule', 'bias_by_epoch_and_rule'),
                    ('Absolute error by epoch and rule', 'absolute_error_by_epoch_and_rule'),
                    ('Prediction diagnostics', 'prediction_diagnostics')]:
    lines += [f'![{title}](figures/{name}.png)', '']
lines += ['## Selected budgets: bias, sample SD, and absolute error', '',
    markdown_table(pd.DataFrame(rows)), '',
    'True theta=(1,-1). Signed bias is the mean estimate minus truth, SD is the sample '
    'standard deviation across repetitions, and MAE is the mean absolute coefficient error. '
    'Nuisance relative MSE is the paper’s squared-error ratio, without a square root. '
    'The first-pass fitting rule is Patience 15 / cap 100.', '',
    '## Monte Carlo uncertainty and limits', '',
    'Ten repetitions provide a broader diagnostic than two, but are still a small Monte Carlo '
    'sample. The CSV summary includes a Monte Carlo standard error for each bias estimate '
    '(sample SD divided by the square root of 10). The paired-drift summary reports the mean '
    'change in absolute error with its Monte Carlo standard error, retaining pairing across '
    'epochs. These describe simulation variability, not model-based coefficient standard '
    'errors or coverage probabilities. Individual paths remain useful because signed errors '
    'can cancel, and the three cases should not be pooled as one common data-generating model.', '',
    'This study holds depth=2, width=32, learning rate=0.005, batch size=128, n=1000, '
    'tau=0.5, and the training/validation split fixed. Each repetition uses a fresh dataset '
    'and its own initialization; it does not separate sampling variability from initialization '
    'variability. The duration comparison is paired within each repetition. True coefficients '
    'and test results are used only for diagnostics; validation loss selects checkpoints. '
    'The 1000-epoch terminal branch deliberately continues beyond early stopping. '
    'A full 200-repetition experiment with paper tuning remains separate. No auxiliary '
    'projection fits, coefficient SEs, coverage, or R stage are included.', '',
    '## Reproduction checks and files', '',
    f'- The {cached_count} reused trajectories retain the original epoch diagnostics after provenance checks.',
    f'- All {verified_original} existing cap-100/patience-15 baseline fits are checked against the saved '
    'first-pass coefficients, nuisance errors, prediction losses, and stopping epochs.',
    '- New repetitions use the same original data generator and deterministic seed sequence; '
    'they have no pre-existing first-pass CSV against which to claim baseline reproduction.',
    '- The training observer checks that it leaves Torch RNG unchanged and clips only an evaluation clone.',
    '- [Source notebook](epoch_robustness.ipynb) / [executed notebook](executed_epoch_robustness.ipynb).',
    '- [Epoch paths](epoch_trajectories.csv), [budget results](epoch_budget_results.csv), '
    '[summary with Monte Carlo SE](epoch_summary.csv), [bias and SD](coefficient_bias_sd_by_epoch.csv).',
    '- [Paired changes](paired_drift_100_to_final.csv), [paired-drift summary](paired_drift_summary.csv), '
    '[baseline checks](baseline_reproduction_checks.csv), [configuration](epoch_run_config.json).',
    '- Four figures are saved in PNG and vector PDF format in `figures/`.', '']
report = '\n'.join(lines)
(OUTPUT_DIR/'RESULTS.md').write_text(report, encoding='utf-8')
display(Markdown(report.split('## Figures')[0]))
print('Report:', OUTPUT_DIR/'RESULTS.md')